In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from pathlib import Path
from resort_ranking.pipeline import build_model
from resort_ranking.data_io import load_matrix, parse_rosstat_csv
from resort_ranking.regions import canonical_region, split_series

RAW = Path("../data/raw")
resorts = pd.DataFrame(load_matrix(RAW / "resorts.json"))
trips_rows = parse_rosstat_csv(RAW / "rosstat_tourist_trips.csv")

rosstat_names = {split_series(r["series"])[0] for r in trips_rows}
canon_to_rosstat = {canonical_region(n): n for n in rosstat_names}

d = (resorts.assign(region_key=resorts["region"].map(canonical_region))
     .groupby("region_key")
     .agg(resort_region_name=("region", "first"), n_resorts=("resort_id", "count"))
     .reset_index())
d["rosstat_region_name"] = d["region_key"].map(canon_to_rosstat)
d["match_status"] = d["rosstat_region_name"].notna().map({True: "найден", False: "НЕ НАЙДЕН"})

assert (d["match_status"] == "найден").all(), "есть регионы без данных Росстата"
d.to_csv("../data/reference/region_dictionary.csv", index=False, encoding="utf-8-sig")
d

,region_key,resort_region_name,n_resorts,rosstat_region_name,match_status
0,Иркутская область,Иркутская область,1,Иркутская область,найден
1,Кабардино-Балкарская Республика,Кабардино-Балкарская Республика,2,Кабардино-Балкарская Республика,найден
2,Камчатский край,Камчатский край,1,Камчатский край,найден
3,Карачаево-Черкесская Республика,Карачаево-Черкесская Республика,2,Карачаево-Черкесская Республика,найден
4,Кемеровская область,Кемеровская область,1,Кемеровская область - Кузбасс,найден
5,Краснодарский край,Краснодарский край,3,Краснодарский край,найден
6,Красноярский край,Красноярский край,1,Красноярский край,найден
7,Ленинградская область,Ленинградская область,1,Ленинградская область,найден
8,Московская область,Московская область,1,Московская область,найден
9,Мурманская область,Мурманская область,3,Мурманская область,найден


In [2]:
model = build_model()
reg = pd.DataFrame(model["regions"])

rows = []
for _, r in reg.iterrows():
    for year in (2022, 2023, 2024, 2025):
        rows.append({"region_key": r["region"], "year": year,
                     "trips": r.get(f"trips_{year}"), "nights": r.get(f"nights_{year}")})
ym = pd.DataFrame(rows)
ym["avg_stay_nights"] = ym["nights"] / ym["trips"]

assert not ym.duplicated(["region_key", "year"]).any()
assert (ym["trips"] > 0).all()
assert (ym.loc[ym["year"] >= 2023, "nights"] > 0).all()

Path("../data/prepared").mkdir(parents=True, exist_ok=True)
ym.to_csv("../data/prepared/region_year_metrics.csv", index=False, encoding="utf-8-sig")
ym.shape   # ожидаем (80, 5)

(80, 5)

In [3]:
wide = ym.pivot(index="region_key", columns="year", values="trips")
flags = pd.DataFrame({
    "growth_2025_%": ((wide[2025] / wide[2024] - 1) * 100).round(1),
    "growth_2024_%": ((wide[2024] / wide[2023] - 1) * 100).round(1),
    "growth_2023_%": ((wide[2023] / wide[2022] - 1) * 100).round(1),
})
flags["clipped_2025"] = flags["growth_2025_%"].abs() > 25
flags["volatile"] = (flags[["growth_2025_%", "growth_2024_%"]].abs() > 40).any(axis=1)
flags = flags.join(d.set_index("region_key")["n_resorts"])
flags.to_csv("../data/prepared/region_flags.csv", encoding="utf-8-sig")
flags.sort_values("growth_2025_%")

,growth_2025_%,growth_2024_%,growth_2023_%,clipped_2025,volatile,n_resorts
region_key,,,,,,
Республика Северная Осетия — Алания,-36.3,22.9,3.1,True,False,2
Республика Алтай,-17.6,3.1,102.3,False,False,2
Хабаровский край,-17.2,6.7,119.3,False,False,1
Кабардино-Балкарская Республика,-16.9,-56.9,174.3,False,True,2
Самарская область,-16.1,32.0,-6.5,False,False,1
Мурманская область,-13.1,21.0,10.0,False,False,3
Кемеровская область,-12.6,23.5,3.8,False,False,1
Иркутская область,-6.9,14.9,62.7,False,False,1
Краснодарский край,-3.8,-0.7,-4.1,False,False,3


In [4]:
m = pd.DataFrame(build_model()["model_rows"])
w = 10 / 100   # вес роста поездок
m["score_no_growth"] = m["popularity_score_default"] - m["region_growth_score"] * w + 50 * w
m["rank_no_growth"] = m["score_no_growth"].rank(ascending=False, method="min").astype(int)
m["rank_shift"] = m["rank_default"] - m["rank_no_growth"]

m[["name_ru", "region", "region_growth_score", "rank_default", "rank_no_growth", "rank_shift"]].head(15)

,name_ru,region,region_growth_score,rank_default,rank_no_growth,rank_shift
0,Газпром Поляна,Краснодарский край,42.321531,1,1,0
1,Красная Поляна,Краснодарский край,42.321531,2,2,0
2,Роза Хутор,Краснодарский край,42.321531,3,3,0
3,Сорочаны,Московская область,99.100406,4,4,0
4,Архыз,Карачаево-Черкесская Республика,100.000000,5,7,-2
5,Домбай,Карачаево-Черкесская Республика,100.000000,6,9,-3
6,Игора,Ленинградская область,48.183896,7,6,1
7,Шерегеш,Кемеровская область,24.742757,8,5,3
8,Горный воздух,Сахалинская область,90.538702,9,14,-5
9,Бобровый лог,Красноярский край,45.332750,10,10,0


In [5]:
m[["resort_id", "name_ru", "rank_default", "rank_no_growth", "rank_shift"]].to_csv(
    "../data/prepared/sensitivity_no_growth.csv", index=False, encoding="utf-8-sig")

In [6]:
import openpyxl

XLSX = "../data/raw/Turpotok_1kv-2026.xlsx"
wb = openpyxl.load_workbook(XLSX, data_only=True)

def read_series(sheet, annual=False):
    rows = list(wb[sheet].iter_rows(values_only=True))
    header = rows[5]
    col = 1 if annual else next(i for i, h in enumerate(header)
                                if h and str(h).strip().startswith("январь-декабрь"))
    return {canonical_region(str(r[0]).strip()): r[col]
            for r in rows[6:] if r[0] and isinstance(r[col], (int, float))}

spec = [("trips",  "annual",    "1.1", (2022, 2023, 2024, 2025), True),
        ("nights", "annual",    "1.2", (2022, 2023, 2024, 2025), True),
        ("trips",  "quarterly", "2.1", (2022, 2023, 2024, 2025), False),
        ("nights", "quarterly", "2.2", (2023, 2024, 2025),       False),
        ("trips",  "monthly",   "3",   (2022, 2023, 2024, 2025), False)]

rows = []
for indicator, release, prefix, years, annual in spec:
    for y in years:
        sheet = f"{prefix}.{y}"
        for region, value in read_series(sheet, annual).items():
            rows.append(dict(region_key=region, indicator=indicator, release=release,
                             year=y, value=value, source_sheet=sheet))
series = pd.DataFrame(rows)
series.to_csv("../data/prepared/rosstat_regional_series.csv", index=False, encoding="utf-8-sig")

# что лежит в проекте
reg = pd.DataFrame(build_model()["regions"])
keys = set(reg["region"])
mine = pd.concat([
    reg.melt(id_vars="region", value_vars=[f"trips_{y}" for y in (2022, 2023, 2024, 2025)]).assign(indicator="trips"),
    reg.melt(id_vars="region", value_vars=[f"nights_{y}" for y in (2023, 2024, 2025)]).assign(indicator="nights"),
])
mine["year"] = mine["variable"].str[-4:].astype(int)
mine = mine.rename(columns={"region": "region_key", "value": "project_value"})[
    ["region_key", "indicator", "year", "project_value"]]

wide = (series[series.region_key.isin(keys)]
        .pivot_table(index=["region_key", "indicator", "year"], columns="release", values="value")
        .reset_index())
cmp = mine.merge(wide, on=["region_key", "indicator", "year"], how="left")
for rel in ["annual", "quarterly", "monthly"]:
    cmp[f"совпадает_{rel}"] = cmp["project_value"] == cmp[rel]
    cmp[f"разница_{rel}_%"] = ((cmp["project_value"] / cmp[rel] - 1) * 100).round(1)

cmp.to_csv("../data/reference/region_manual_check.csv", index=False, encoding="utf-8-sig")
cmp.groupby("indicator")[["совпадает_annual", "совпадает_quarterly", "совпадает_monthly"]].sum()

,совпадает_annual,совпадает_quarterly,совпадает_monthly
indicator,,,
nights,0,60,0
trips,0,0,80


In [7]:
   t = series[series.indicator == "trips"].pivot_table(
       index=["region_key", "release"], columns="year", values="value")
   t["рост_2025_%"] = ((t[2025] / t[2024] - 1) * 100).round(1)
   check3 = t.loc[["Карачаево-Черкесская Республика", "Московская область", "Краснодарский край"],
                  [2024, 2025, "рост_2025_%"]]
   check3.to_csv("../data/reference/growth_check_3_regions.csv", encoding="utf-8-sig")
   check3

year                                             2024        2025  рост_2025_%
region_key                      release                                       
Карачаево-Черкесская Республика annual       742180.0    966509.0         30.2
                                monthly      699802.0   1303991.0         86.3
                                quarterly    528737.0    868973.0         64.3
Московская область              annual     22830278.0  20876829.0         -8.6
                                monthly    18009855.0  22431311.0         24.6
                                quarterly  22605867.0  20265690.0        -10.4
Краснодарский край              annual     20120419.0  19558981.0         -2.8
                                monthly    19163040.0  18427326.0         -3.8
                                quarterly  19497128.0  19781492.0          1.5